# Embedding

Now you have loaded your data, it's time to build embeddings over the files so you can start querying over them.

## What is an Embedding

`Vector Embeddings`, often referred just as embeddings, are numerical representations that capture the meanings and relationships of words, phrases, and other data. They convert important features of an object into a simple array of numbers, making it easy for computers to find information quickly. Similar data points are placed close together in a multidimensional space, making them easy to group and analyze.

These mathematical representations enables `semantic search` where user provides a query and Langdb can locate the text related to the meaning of the query rather than just keyword matching. 

Embeddings are a big part of how `Retrieval-Augmented Generation (RAG)` works.  

There are various types of embeddings, each differing in efficiency, effectiveness, and computational cost. Langdb, by default, uses `sentence-transformers/all-MiniLM-L6-v2`, a sentence-transformer model on HuggingFace.

## How you can use LangDB to generate Vector Embeddings

LangDB, allows you to seamlessly generate embeddings using SQL. These embeddings embeddings can be stored inside Clickhouse for searching.

To generate embeddings, pass the content of your data into [`embed`](https://langdb.ai/docs/sql_functions/embed/) function:

In [ ]:
CREATE TABLE pdf_embeddings (
  id UUID,
  embeddings `Array`(`Float32`),
) 
engine = MergeTree
order by id;

In [ ]:
INSERT INTO pdf_embeddings
select p.id, embed(content) from pdfs;

With your PDF content embedded, the data is now technically ready to for querying. However, embedding all your text can be time-consuming, so it is advisable to create a recurring [task](https://langdb.ai/docs/sql_statements/spawn_task/) for it.

In [ ]:
SPAWN TASK generate_embeddings
    BEGIN
INSERT INTO pdf_embeddings
select p.id, embed(content) from pdfs as p LEFT JOIN pdf_embeddings as pe on p.id = pe.id
where p.id != pe.id
order by p.id
limit 10
END EVERY 5 second;

In [1]:
This will generate 10 embeddings every 5 seconds and store it inside `pdf_embeddings` table. 